# transformer_ko6 — knockout transformer + mRNA/regulatory layer (Colab / GPU)

**v6** = v5 (interaction-token transformer trained as a metric encoder on **6 Perturb-seq lines + Replogle**, distillation,
retrieval readout) **+ the mRNA/regulatory layer**: regulatory token features (is-TF, in/out-degree from TRRUST + 60k
SIGNOR/CollecTRI + K562 ChIP) and a direct TF→target channel. Scored on held-out K562, tide-removed specific-mover recall@50.

It reports the **TF-KO subset separately** — the honest place the direct channel can fire (only ~23% of KOs are regulatory
sources). CPU smoke here gave: v5 0.506 → +reg-features 0.517 → +channel 0.517 (β tuned to 0 = channel adds nothing).
The GPU 3-seed run confirms whether that +0.011 from features is stable.

### Data (fast — no Drive search)
All Perturb-seq lines + regulatory JSONs ship **in the repo clone**. Only two large artifacts come from Drive, at **exact
paths** (from your earlier runs): `MyDrive/cell_model/cell_complete.json` and
`MyDrive/cell_model/artifacts/depmap_vecs.npz`. Set **Runtime → GPU**.


In [ ]:
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
!pip -q install scipy >/dev/null 2>&1


In [ ]:
# clone / update to latest branch head (brings colab/*.py + all nlz_*.pkl + regulatory JSONs)
import os
BRANCH = 'claude/vectorize-gex-propensity-zp09w8'
if not os.path.isdir('/content/cell'):
    !git clone --branch $BRANCH --depth 1 https://github.com/Nikku03/cell.git /content/cell
else:
    !cd /content/cell && git fetch --depth 1 origin $BRANCH && git reset --hard origin/$BRANCH
%cd /content/cell
!ls outputs/orphan/nlz_*.pkl | wc -l && echo 'nlz lines present'


In [ ]:
# mount Drive and copy the 2 big artifacts from EXACT paths (no recursive glob -> instant)
from google.colab import drive
import shutil, os
drive.mount('/content/drive')
SP = '/tmp/claude-0/-home-user-cell/0f039315-b3a9-52ac-8187-9fae0d726994/scratchpad'   # eval_harness / transformer_ko5.profiles look here for nlz pickles
os.makedirs(SP, exist_ok=True); os.makedirs('/content/cell/outputs/orphan', exist_ok=True)

# exact Drive locations (adjust ONLY if yours differ):
DRIVE = {
  '/content/drive/MyDrive/cell_model/cell_complete.json'          : '/content/cell/outputs/orphan/cell_complete.json',
  '/content/drive/MyDrive/cell_model/artifacts/depmap_vecs.npz'   : '/content/cell/outputs/orphan/depmap_vecs.npz',
}
for src, dst in DRIVE.items():
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        print('present:', os.path.basename(dst)); continue
    assert os.path.exists(src), f'NOT at expected path: {src}\n  -> fix the path above to where your file actually is.'
    shutil.copy(src, dst); print(f'copied {os.path.basename(dst)}  <-  {src}')

# the nlz pickles ship in the repo -> copy them to the scratch path the code expects (fast, local)
for L in ['K562','RPE1','HepG2','Jurkat','HCT116','Melanoma','Replogle']:
    r = f'/content/cell/outputs/orphan/nlz_{L}.pkl'; d = f'{SP}/nlz_{L}.pkl'
    if os.path.exists(r) and not os.path.exists(d): shutil.copy(r, d)
print('nlz staged:', sum(os.path.exists(f'{SP}/nlz_{L}.pkl') for L in ['K562','RPE1','HepG2','Jurkat','HCT116','Melanoma','Replogle']), '/ 7')
print('all artifacts staged.')


In [ ]:
# (optional) fast GPU smoke: 1 seed, few epochs (~1-2 min)
!cd /content/cell && V6_SMOKE=1 python colab/transformer_ko6.py


In [ ]:
# FULL run: 3 splits, v5(no reg) vs +reg-features vs +direct-channel, + TF-KO subset breakdown
!cd /content/cell && python colab/transformer_ko6.py


### Reading the output
- **`v5 (no reg)`** — the multi-line+Replogle metric encoder, no regulatory layer (the base).
- **`+reg-features`** — regulatory token features added (is-TF, in/out-degree).
- **`+reg-features +channel`** — plus the direct TF→target fusion (β tuned on val; β=0 means val chose to ignore it).
- **`TF-KO subset`** — recall restricted to the ~23% of test KOs that are regulatory sources — where the direct channel *could* help.
- **`tide` ~0.25** floor, **`oracle` ~0.61** retrieval ceiling.

If `+channel` ≈ `+reg-features` and β→0, the direct TF→target channel is null on this KO panel (thin coverage: median 2 measured
targets per source). The regulatory *features* giving a small stable + is the honest signal to watch.
